# Analytical Notebook — Customer Churn

**Project:** Customer Behaviour & Business Performance Analysis  
**Company:** ABC Communications Ltd  
**Programme:** AnalystLab Africa — Data Analytics Internship Programme  
**Prepared by:** **Nancy Lee YIMBERE ALAPINI**  
*Performance & Decision Intelligence Analyst | KPI Design, Performance Management & Decision Support | Telecom QoS • Human & Operational Performance*

---

## 1. Notebook Objective

This notebook transparently documents the analytical workflow actually used for the project.

- **Power BI** was the primary environment for data preparation, KPI development, customer segmentation and business visualisation.
- **Python** was used as a complementary tool for the **Box Plot** and **Correlation Heatmap**.
- The notebook serves as a **technical and reproducible record** of the project; it does not imply that the entire analysis was originally performed in Python.

The dataset contains **7,043 customers and 21 variables**.

## 2. Analytical Workflow

**Source dataset → Power BI → data control/preparation → KPIs & segmentation → Python for complementary statistical visualisations → interpretation → recommendations.**

The `customerID` field is used as the customer key. The business target is `Churn` (`Yes` / `No`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(DATA_PATH)

print("Dimensions :", df.shape)
df.head()

## 3. Inspection du dataset

Les contrôles documentés dans le Dataset Inspection Report portent sur :
- dimensions ;
- types de données ;
- valeurs manquantes / chaînes vides ;
- doublons ;
- statistiques descriptives.

Une attention particulière est portée à `TotalCharges` : dans le CSV source, 11 valeurs sont des chaînes vides. Elles correspondent toutes à des clients avec `tenure = 0`.

In [ ]:
# Dimensions, doublons et valeurs vides
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Duplicated rows:", df.duplicated().sum())
print("Duplicated customerID:", df["customerID"].duplicated().sum())

blank_totalcharges = df["TotalCharges"].astype(str).str.strip().eq("")
print("Blank TotalCharges:", blank_totalcharges.sum())
print(df.loc[blank_totalcharges, ["customerID", "tenure", "TotalCharges"]])

In [ ]:
# Conversion analytique de TotalCharges
df["TotalCharges_num"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

df[["tenure", "MonthlyCharges", "TotalCharges_num"]].describe().round(2)

## 3. Dataset Inspection

The controls documented in the Dataset Inspection Report cover:
- dimensions;
- data types;
- missing values / blank strings;
- duplicates;
- descriptive statistics.

Particular attention is given to `TotalCharges`: in the source CSV, 11 values are blank strings. All correspond to customers with `tenure = 0`.

## 4. KPI principaux — calculés dans Power BI

Les KPI de synthèse utilisés dans le dashboard sont :
- **Total Customers**
- **Churned Customers**
- **Churn Rate**
- **Retention Rate**

Le bloc Python ci-dessous reproduit uniquement les résultats afin de rendre le notebook vérifiable.

In [ ]:
total_customers = df["customerID"].nunique()
churned_customers = df.loc[df["Churn"].eq("Yes"), "customerID"].nunique()
churn_rate = churned_customers / total_customers
retention_rate = 1 - churn_rate

pd.Series({
    "Total Customers": total_customers,
    "Churned Customers": churned_customers,
    "Churn Rate": churn_rate,
    "Retention Rate": retention_rate
})

**Résultats du projet :** 7 043 clients  1 869 churned customers  **26.54% de churn** et **73.46% de rétention**.

### Data Preparation Decision

The 11 affected customer records are **retained**. Their `tenure = 0` provides a coherent business interpretation: these customers are at the beginning of the customer relationship and have not yet accumulated historical charges. Missing `TotalCharges` values are therefore handled explicitly in calculations requiring this variable rather than automatically deleting the customers.

In [ ]:
# Churn Rate by Contract Type
(df.assign(ChurnFlag=df["Churn"].eq("Yes").astype(int))
   .groupby("Contract")["ChurnFlag"].mean()
   .mul(100).round(2)
   .sort_values(ascending=False))

In [ ]:
# Churn Rate by Tenure Group
bins = [-1, 12, 24, 48, 72]
labels = ["0–12 months", "13–24 months", "25–48 months", "49–72 months"]
df["TenureGroup"] = pd.cut(df["tenure"], bins=bins, labels=labels)

(df.assign(ChurnFlag=df["Churn"].eq("Yes").astype(int))
   .groupby("TenureGroup", observed=True)["ChurnFlag"].mean()
   .mul(100).round(2))

In [ ]:
# Churn Rate by Internet Service
(df.assign(ChurnFlag=df["Churn"].eq("Yes").astype(int))
   .groupby("InternetService")["ChurnFlag"].mean()
   .mul(100).round(2)
   .sort_values(ascending=False))

## 4. Core KPIs — Developed in Power BI

The summary KPIs used in the dashboard are:
- **Total Customers**
- **Churned Customers**
- **Churn Rate**
- **Retention Rate**

The Python block below reproduces the results solely to make the notebook verifiable.

In [ ]:
def churn_rate(mask):
    segment = df.loc[mask]
    return round(segment["Churn"].eq("Yes").mean() * 100, 2)

all_customers = pd.Series(True, index=df.index)
m1 = df["Contract"].eq("Month-to-month")
m2 = m1 & df["InternetService"].eq("Fiber optic")
m3 = m2 & df["PaymentMethod"].eq("Electronic check")
m4 = m3 & df["tenure"].between(0, 12)

pd.Series({
    "Overall population": churn_rate(all_customers),
    "Month-to-month": churn_rate(m1),
    "+ Fiber Optic": churn_rate(m2),
    "+ Electronic Check": churn_rate(m3),
    "+ Tenure 0–12 months": churn_rate(m4)
}, name="Churn Rate (%)")

## 7. Analyse TechSupport / OnlineSecurity — Fiber Optic

L’analyse Power BI a approfondi le segment Fiber Optic en comparant les configurations de `TechSupport` et `OnlineSecurity`. Cette étape permet d’identifier une piste d’action métier à tester, sans conclure à la causalité.

In [ ]:
fiber = df[df["InternetService"].eq("Fiber optic")].copy()

fiber["SupportSecurity"] = np.select(
    [
        fiber["TechSupport"].eq("Yes") & fiber["OnlineSecurity"].eq("Yes"),
        fiber["TechSupport"].eq("Yes") & ~fiber["OnlineSecurity"].eq("Yes"),
        ~fiber["TechSupport"].eq("Yes") & fiber["OnlineSecurity"].eq("Yes")
    ],
    ["Both", "TechSupport only", "OnlineSecurity only"],
    default="Neither"
)

(fiber.assign(ChurnFlag=fiber["Churn"].eq("Yes").astype(int))
      .groupby("SupportSecurity")["ChurnFlag"].mean()
      .mul(100).round(2)
      .sort_values(ascending=False))

**Project results:** 7,043 customers, 1,869 churned customers, **26.54% churn rate** and **73.46% retention rate**.

In [ ]:
no_churn = df.loc[df["Churn"].eq("No"), "MonthlyCharges"]
yes_churn = df.loc[df["Churn"].eq("Yes"), "MonthlyCharges"]

fig, ax = plt.subplots(figsize=(7, 4.5))
bp = ax.boxplot([no_churn, yes_churn], tick_labels=["No", "Yes"], patch_artist=True)

bp["boxes"][0].set_facecolor("#3B82F6")
bp["boxes"][1].set_facecolor("#F59E0B")

ax.set_title("Monthly Charges by Churn Status")
ax.set_xlabel("Churn")
ax.set_ylabel("Monthly Charges")
plt.show()

df.groupby("Churn")["MonthlyCharges"].agg(["mean", "median"]).round(2)

## 5. Business Segmentation Performed in Power BI

Three segmentations structure the main diagnosis:
1. contract type;
2. tenure;
3. Internet service.

The calculations below provide a reproducible control of the rates presented in Power BI.

In [ ]:
df["Churn_num"] = df["Churn"].map({"No": 0, "Yes": 1})

corr_cols = ["tenure", "MonthlyCharges", "TotalCharges_num", "SeniorCitizen", "Churn_num"]
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(7.5, 5.5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)

ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr.index)), corr.index)

for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")

fig.colorbar(im, ax=ax)
ax.set_title("Correlation Heatmap of Numerical Variables")
plt.tight_layout()
plt.show()

corr.round(2)

## 10. Résultats analytiques clés

Les principaux résultats consolidés du projet sont :

- Churn global : **26.54%**
- Month-to-month : **42.71%**
- Tenure 0–12 months : **47.44%**
- Fiber Optic : **41.89%**
- Profil Month-to-month + Fiber Optic : **54.61%**
- + Electronic Check : **60.37%**
- + Tenure 0–12 months : **71.16%**
- Fiber Optic sans TechSupport ni OnlineSecurity : **60.70%**
- Fiber Optic avec TechSupport + OnlineSecurity : **28.05%**

Ces résultats soutiennent une stratégie de rétention **ciblée, précoce et mesurable**.

## 11. Limites d’interprétation

1. Les résultats montrent des **associations**, pas des causalités.
2. Le dataset ne contient pas toutes les variables opérationnelles susceptibles d’expliquer l’expérience Fiber Optic (incidents, qualité réseau, réclamations, etc.).
3. `TotalCharges` est fortement lié à l’ancienneté par construction.
4. Les recommandations de migration contractuelle ou de bundle doivent être **testées** avant généralisation.
5. Le taux de churn observé dans un segment ne constitue pas à lui seul une preuve qu’une caractéristique du segment provoque le churn.

## 12. Conclusion technique

Le workflow hybride **Power BI + Python** permet de séparer clairement les rôles :

- **Power BI** : préparation, KPI, segmentation, diagnostic métier et dashboard décisionnel ;
- **Python** : visualisations statistiques complémentaires et contrôle des relations numériques ;
- **Business Analytics Report** : consolidation des insights, risques, opportunités et recommandations.

Ce notebook constitue la trace technique du projet et complète le Dataset Inspection Report, le Business Analytics Report et la présentation finale.